In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense

In [3]:
sentences = [
    "I eat an apple every morning",
    "I eat a banana every morning",
    "I eat an orange every morning",
    "I like fresh apples",
    "I like fresh bananas",
    "I like fresh oranges",
    "I eat fresh fruit",
    "I like fruit every morning",
    "apples and bananas are healthy",
    "oranges and apples are healthy"
]

In [4]:
tokenizer = Tokenizer()

tokenizer.fit_on_texts(sentences)

word_index = tokenizer.word_index

print(word_index)

{'i': 1, 'eat': 2, 'every': 3, 'morning': 4, 'like': 5, 'fresh': 6, 'apples': 7, 'an': 8, 'bananas': 9, 'oranges': 10, 'fruit': 11, 'and': 12, 'are': 13, 'healthy': 14, 'apple': 15, 'a': 16, 'banana': 17, 'orange': 18}


In [5]:
sequences = tokenizer.texts_to_sequences(sentences)

for sentence, sequence in zip(sentences, sequences):
    print(sentence)
    print(sequence)
    print()

I eat an apple every morning
[1, 2, 8, 15, 3, 4]

I eat a banana every morning
[1, 2, 16, 17, 3, 4]

I eat an orange every morning
[1, 2, 8, 18, 3, 4]

I like fresh apples
[1, 5, 6, 7]

I like fresh bananas
[1, 5, 6, 9]

I like fresh oranges
[1, 5, 6, 10]

I eat fresh fruit
[1, 2, 6, 11]

I like fruit every morning
[1, 5, 11, 3, 4]

apples and bananas are healthy
[7, 12, 9, 13, 14]

oranges and apples are healthy
[10, 12, 7, 13, 14]



In [6]:
window_size = 2

X = []
y = []

for sequence in sequences:
    for i in range(window_size, len(sequence) - window_size):
        context = (
            sequence[i-window_size:i] +
            sequence[i+1:i+window_size+1]
        )

        target = sequence[i]

        X.append(context)
        y.append(target)

X = np.array(X)
y = np.array(y)
print(X, y)

[[ 1  2 15  3]
 [ 2  8  3  4]
 [ 1  2 17  3]
 [ 2 16  3  4]
 [ 1  2 18  3]
 [ 2  8  3  4]
 [ 1  5  3  4]
 [ 7 12 13 14]
 [10 12 13 14]] [ 8 15 16 17  8 18 11  9  7]


In [14]:
vocab_size = len(word_index) + 1

embedding_dim = 3

In [15]:
model = Sequential([
    Embedding(
        input_dim =vocab_size,
        output_dim = embedding_dim,
        input_length = X.shape[1]
    ),
    tf.keras.layers.GlobalAveragePooling1D(), #average
    Dense(
        vocab_size,
        activation="softmax"
    )
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X, y, epochs=100, batch_size=16, verbose=1
)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 402ms/step - accuracy: 0.1111 - loss: 2.9456
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.1111 - loss: 2.9441
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1111 - loss: 2.9425
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2222 - loss: 2.9410
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.2222 - loss: 2.9394
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.2222 - loss: 2.9379
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2222 - loss: 2.9363
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.2222 - loss: 2.9348
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.2222 - loss: 2.9333
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.2222 - loss: 2.9317
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.2222 - loss: 2.9302
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2222 - l

In [16]:
embedding_matrix = model.layers[0].get_weights()[0]
print(embedding_matrix)

[[-0.03853573 -0.02299145 -0.00696987]
 [-0.0936778  -0.07006468 -0.0742415 ]
 [-0.16455118 -0.11833857 -0.13852558]
 [-0.12821573 -0.1186071  -0.04817571]
 [-0.14634407 -0.1733384   0.14349024]
 [-0.04450341 -0.08378294  0.11494928]
 [ 0.03564912 -0.03843039  0.04712358]
 [-0.04832306 -0.03914281 -0.05721544]
 [-0.03182449 -0.1561634   0.07278678]
 [ 0.02166175  0.02181182 -0.00298448]
 [ 0.10948614  0.1405083   0.11934665]
 [-0.04270166 -0.04874883 -0.02156711]
 [ 0.08917768  0.03380981 -0.01111288]
 [ 0.07757597  0.07821509  0.00876609]
 [ 0.12217647  0.05577883  0.01492694]
 [-0.09222928 -0.0611794  -0.11826333]
 [-0.12563582 -0.11846054  0.07394315]
 [ 0.13004196  0.07432926 -0.08243815]
 [-0.16346031 -0.15284245 -0.14963666]]


In [17]:
for word, index in word_index.items():
    vector = embedding_matrix[index]
    print(word)
    print(vector)

i
[-0.0936778  -0.07006468 -0.0742415 ]
eat
[-0.16455118 -0.11833857 -0.13852558]
every
[-0.12821573 -0.1186071  -0.04817571]
morning
[-0.14634407 -0.1733384   0.14349024]
like
[-0.04450341 -0.08378294  0.11494928]
fresh
[ 0.03564912 -0.03843039  0.04712358]
apples
[-0.04832306 -0.03914281 -0.05721544]
an
[-0.03182449 -0.1561634   0.07278678]
bananas
[ 0.02166175  0.02181182 -0.00298448]
oranges
[0.10948614 0.1405083  0.11934665]
fruit
[-0.04270166 -0.04874883 -0.02156711]
and
[ 0.08917768  0.03380981 -0.01111288]
are
[0.07757597 0.07821509 0.00876609]
healthy
[0.12217647 0.05577883 0.01492694]
apple
[-0.09222928 -0.0611794  -0.11826333]
a
[-0.12563582 -0.11846054  0.07394315]
banana
[ 0.13004196  0.07432926 -0.08243815]
orange
[-0.16346031 -0.15284245 -0.14963666]


In [19]:
word = "orange"
index = word_index[word]
vector = embedding_matrix[index]

print(word)
print(vector)

orange
[-0.16346031 -0.15284245 -0.14963666]
